# How To: Correct Edge Effects in Plate Assays

Colonies on the outer rows and columns of an agar plate often grow
differently due to temperature gradients and humidity effects. Use
`EdgeCorrector` to statistically identify and correct these biases.

In [1]:
from phenotypic.data import load_meas
from phenotypic.analysis import EdgeCorrector

## Load Measurement Data

`EdgeCorrector` operates on measurement DataFrames, not on images
directly. Start with a DataFrame containing colony measurements and
grid position information.

In [2]:
df = load_meas()
print(f"Shape: {df.shape}")
df.head()

Shape: (129, 245)


,Metadata_Condition,Metadata_Media,Metadata_Set,Metadata_Replicate,Metadata_Time,Metadata_Strain,Metadata_FileName,ObjectLabel,Bbox_CenterRR,Bbox_CenterCC,...,TextureGray_InfoCorrelation1-deg135-scale04,TextureGray_InfoCorrelation2-deg000-scale04,TextureGray_InfoCorrelation2-deg045-scale04,TextureGray_InfoCorrelation2-deg090-scale04,TextureGray_InfoCorrelation2-deg135-scale04,Metadata_StrainID,TextureGray_HaralickVariance-avg-scale04,CorrectedCarryingCapacity_Intensity,CorrectedCarryingCapacity_Area,Metadata_Dataset
0,30C,S,3,4,24,CBS11445,30C_2_3S_4,14,263.025755,327.103517,...,-0.033962,0.523065,0.263252,0.538240,0.318621,220,0.662881,1629.588639,4183.333333,S 30C
1,30C,S,3,5,24,CBS11445,30C_2_3S_5,14,309.120466,291.332736,...,-0.035568,0.605276,0.243759,0.563349,0.376115,220,1.373371,1433.869174,3225.333333,S 30C
2,30C,S,3,6,24,CBS11445,30C_2_3S_6,13,312.525658,262.907288,...,-0.068525,0.733840,0.519368,0.658842,0.497267,220,1.298390,1206.170181,2984.666667,S 30C
3,30C,S,3,12,24,CBS11445,30C_2_3S_12,16,296.079598,305.151655,...,-0.057264,0.715456,0.470510,0.672158,0.475346,220,1.870195,1279.117711,3074.666667,S 30C
4,30C,S,3,8,24,CBS11445,30C_2_3S_8,14,318.208931,260.158426,...,-0.034110,0.589887,0.358457,0.596423,0.342946,220,1.067527,1244.967859,2988.333333,S 30C


## Apply Edge Correction

Specify the measurement column to correct (`on`), the grouping columns
(`groupby`), and the grid dimensions.

In [3]:
corrector = EdgeCorrector(
    on="Shape_Area",
    groupby=["Metadata_Strain"],
    nrows=8,
    ncols=12,
    top_n=3,
    pvalue=0.05,
)

In [4]:
corrected = corrector.analyze(df)
corrected.head()

,Metadata_Strain,Grid_RowMajorIdx,Metadata_Time,Metadata_Condition,Metadata_Media,Metadata_Set,Metadata_Replicate,Metadata_FileName,ObjectLabel,Bbox_CenterRR,...,TextureGray_InfoCorrelation2-deg045-scale04,TextureGray_InfoCorrelation2-deg090-scale04,TextureGray_InfoCorrelation2-deg135-scale04,Metadata_StrainID,TextureGray_HaralickVariance-avg-scale04,CorrectedCarryingCapacity_Intensity,CorrectedCarryingCapacity_Area,Metadata_Dataset,EdgeCorrection_NewVal-Shape_Area,EdgeCorrection_Cap-Shape_Area
0,CBS11445,13,24,30C,S,3,4,30C_2_3S_4,14,263.025755,...,0.263252,0.538240,0.318621,220,0.662881,1629.588639,4183.333333,S 30C,2123.000000,3018.454545
1,CBS11445,13,36,30C,S,3,4,30C_3_3S_4,13,281.276198,...,0.511469,0.752012,0.555040,220,3.543490,1629.588639,4183.333333,S 30C,2319.750000,3018.454545
2,CBS11445,13,48,30C,S,3,2,30C_4_3S_2,15,251.863324,...,0.407750,0.679685,0.401600,220,1.761140,1554.083914,4085.333333,S 30C,2526.866667,3018.454545
3,CBS11445,13,60,30C,S,3,2,30C_5_3S_2,18,288.268624,...,0.613829,0.796545,0.611889,220,1.791946,1554.083914,4085.333333,S 30C,3018.454545,3018.454545
4,CBS11445,13,72,30C,S,3,8,30C_6_3S_8,11,226.486570,...,0.491578,0.691649,0.521048,220,2.688246,1244.967859,2988.333333,S 30C,2773.194444,3018.454545


The corrected DataFrame contains adjusted values for edge-affected
colonies. The `top_n` parameter controls how many interior colonies
are used as the reference baseline.